# Make a dataset from a jsonl file

In [17]:
import pandas as pd    
jsonObj = pd.read_json(path_or_buf="../records_export_full/stift-heiligenkreuz__2026-08-05T06-46-32Z.jsonl", lines=True)

# get only the necessary columns
df = pd.DataFrame(jsonObj)
del df["kp_id"]
del df["src_id"]
del df["institution_id"]
df


,content,pipeline_id
0,{'id': 'https://id.kulturpool.at/3298d5ec-2b53...,stift-heiligenkreuz
1,{'id': 'https://id.kulturpool.at/597b9d95-9e1d...,stift-heiligenkreuz
2,{'id': 'https://id.kulturpool.at/e4c4a5e7-8a38...,stift-heiligenkreuz
3,{'id': 'https://id.kulturpool.at/13cda2e6-2609...,stift-heiligenkreuz
4,{'id': 'https://id.kulturpool.at/500a2ffa-e9d3...,stift-heiligenkreuz
...,...,...
2390,{'id': 'https://id.kulturpool.at/7ae2102b-1e97...,stift-heiligenkreuz
2391,{'id': 'https://id.kulturpool.at/17c5cd2a-4f60...,stift-heiligenkreuz
2392,{'id': 'https://id.kulturpool.at/ed46a3d5-93c2...,stift-heiligenkreuz
2393,{'id': 'https://id.kulturpool.at/ad4c7e71-d403...,stift-heiligenkreuz


In [18]:
mydict = df["content"].to_list()
mydict[:5]

[{'id': 'https://id.kulturpool.at/3298d5ec-2b53-4c3a-8fbb-044356f17ecf/aggregation',
  'type': 'Aggregation',
  'hasView': [{'id': 'https://assets.museum-digital.org/at-noe/images/3/259-0215/entwurf_der_beschriftung_/entwurf-der-beschriftung-des-buchumschlages-fuer-arthur-roesslers-teschner-busch-von-1947-2.jpg',
    'type': 'WebResource',
    'creator': ['Erwin Pokorny'],
    'dcRights': ['Stift Heiligenkreuz'],
    'edmRights': 'http://creativecommons.org/licenses/by-nc-sa/4.0/'}],
  '@context': 'https://api.kulturpool.at/ns/v1/edm.json',
  'provider': 'Kulturpool',
  'edmRights': 'http://creativecommons.org/licenses/by-nc-sa/4.0/',
  'isShownAt': 'https://noe.at.museum-digital.org/object/259',
  'isShownBy': {'id': 'https://assets.museum-digital.org/at-noe/images/3/259-0215/entwurf_der_beschriftung_/entwurf-der-beschriftung-des-buchumschlages-fuer-arthur-roesslers-teschner-busch-von-1947.jpg',
   'type': 'WebResource',
   'creator': ['Erwin Pokorny'],
   'dcRights': ['Stift Heiligen

In [25]:
from collections import Counter

def find_key(data, key):
    """Yield every value stored under `key`, at any depth."""
    if isinstance(data, dict):
        for k, v in data.items():
            if k == key:
                yield v
            else:
                yield from find_key(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_key(item, key)

def unpack(v):
    if isinstance(v, list):
        for item in v:
            yield from unpack(item)
    elif isinstance(v, dict):
        if "@value" in v:
            yield v["@value"], v.get("@language")
    elif v is not None:
        yield v, None

value_counts = Counter()
value_langs = {}

for d in mydict:
    for medium in find_key(d, "medium"):
        for value, lang in unpack(medium):
            value_counts[value] += 1
            value_langs[value] = lang

print(value_counts)
print(value_langs)

Counter({'Papier / Holzschnitt': 129, 'Papier / Lithografie': 112, 'Papier / Radierung': 102, 'Papier / Kupferstich': 101, 'Öl auf Leinwand': 97, 'Terrakotta': 78, 'Papier / Klischeedruck': 36, 'Papier / Bleistift': 34, 'Aquarell, auf Papier.': 33, 'Holz, Draht, Textil (Filz), Wachs, Perlen': 33, 'Bleistiftvorzeichnung und Aquarell auf Papier': 30, 'Bleistift auf Papier': 28, 'Pergament, Tinte / Federzeichnung': 25, 'Glas': 24, 'Aquarell auf Karton': 24, 'Öl auf Holz': 22, 'Büttenpapier Somerset/ Aquatinta': 20, 'Papier / Lichtdruck': 20, 'Papier / Lithografie, koloriert': 20, 'Bleistift, auf Transparentpapier.': 19, 'Papier / Chromolithografie': 17, 'Papier / Kupferstich, koloriert': 16, 'Zinn - Zinnguss': 16, 'Holz': 16, 'Papier / Aquarell; Tusche; Deckweiß': 16, 'Papier / Kaltnadelradierung': 14, 'Papier / Druck': 14, 'Papier / Stahlstich': 14, 'Papier / Holzstich': 14, 'Bleistift, auf Papier.': 14, 'Papier / Linolschnitt': 13, 'Schwarze Tusche, Bleistift auf Maschinenpapier (beige,

In [ ]:
from collections import Counter

def find_values(data, key=None):
    if isinstance(data, dict):
        if "@value" in data and key is None:
            yield data
        for k, v in data.items():
            if key is not None and k == key:
                yield from find_values(v)
            else:
                yield from find_values(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_values(item, key)


value_counts = Counter()
value_langs = {}

for d in mydict:
    for obj in find_values(d, key="medium"):
        value = obj["@value"]
        lang = obj.get("@language")   # not every @value has a language
        value_counts[value] += 1
        value_langs[value] = lang

print(dict(value_counts))
print(value_langs)

{}
{}


In [29]:
data = {"word": value_counts.keys(),
        "count": value_counts.values(),
        #"lang": value_langs.values(),
        }
df = pd.DataFrame(data)
df = df.sort_values("count", ascending=False)
df

,word,count
1,Papier / Holzschnitt,129
33,Papier / Lithografie,112
40,Papier / Radierung,102
21,Papier / Kupferstich,101
197,Öl auf Leinwand,97
...,...,...
686,"Aquarell, schwarze Tusche, schwarze Kreide auf...",1
685,"Kreiden, Aquarellfarben, schwarze Tusche, Deck...",1
684,"Aquarell, Deckweiß, schwarze Tusche, schwarze ...",1
713,"Aquarell, schwarze Tusche, blaue (Öl)kreide, s...",1


In [5]:
df.loc[df['lang'] == "de", 'word']

11086    Papierschneidemaschine
22366              Abgaskrümmer
11280               Guttapercha
11029             Arbeitskasten
10854           Seitenschneider
                  ...          
4564              Prothesenteil
6804          Bestrahlungslampe
2411                Gewichtsatz
4860                 Übertrager
4563             Blattschreiber
Name: word, Length: 5850, dtype: str

In [ ]:
en_list = df.loc[df['lang'] == "de", 'word'].to_list()

In [30]:
stift_list = df["word"].to_list()
print(len(stift_list))

721


In [32]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(stift_list)

clustering = AgglomerativeClustering(
    distance_threshold=0.8,  
    n_clusters=None,
    linkage='ward'
)

clustering.fit(embeddings)

# Step 4: Group words by cluster labels
labels = clustering.labels_
clusters = {}
for word, label in zip(stift_list, labels):
    clusters.setdefault(label, []).append(word)

# Step 5: Print clusters
for label, cluster_words in clusters.items():
    print(f"Cluster {label}: {cluster_words}")
print(len(clusters))

Cluster 36: ['Papier / Holzschnitt', 'Papier / Inkunabel und Holzschnitt', 'Pappe / Holzschnitt']
Cluster 105: ['Papier / Lithografie', 'Papier / Tonlithografie', 'Papier / Farblithografie']
Cluster 21: ['Papier / Radierung', 'Papier / Ätzradierung', 'Papier / Farbradierung', 'Papier / Steinradierung', 'Papier / Radierung (Ätzverfahren)', 'Papier / Umrissradierung']
Cluster 58: ['Papier / Kupferstich', 'Papier, Karton / Kupferstich', 'Papier/ Kupferstich', 'Papier / Buchdruck und Kupferstich', 'Papier / Buchdruck, Kupferstich']
Cluster 55: ['Öl auf Leinwand', 'Öl auf Leinwand; Ochsenaugenrahmen']
Cluster 77: ['Terrakotta', 'Terrakotta - Schlagmetall']
Cluster 0: ['Papier / Klischeedruck', 'Papier', 'Papier / Schabkunst', 'Papier / Kreide', 'Papier / Stich']
Cluster 9: ['Papier / Bleistift', 'Papier (Heft gebunden) / Bleistift', 'Bleistift / Papier']
Cluster 11: ['Aquarell, auf Papier.', 'Aquarell, Gouache, auf Papier.', 'Aquarell auf Papier.', 'Aquarell auf Papier', 'Aquarell auf Papie